# ProgressiveHammingReward vs Standard HammingReward

Standard HammingReward normalizes by `target_len`, so intermediate rewards have poor discrimination at early steps. ProgressiveHammingReward normalizes by `current_len` instead.

| Step | Sequence | Standard (matches/target_len) | Progressive (matches/current_len) |
|------|---------|-------------------------------|-----------------------------------|
| t=1 | `U` (match) | 1/22 = 0.045 | 1/1 = 1.0 |
| t=1 | `A` (mismatch) | 0/22 = 0.0 | 0/1 = 0.0 |
| t=5 | `UGAGG` (match) | 5/22 = 0.227 | 5/5 = 1.0 |
| t=5 | `ACGUU` (random) | ~1/22 = 0.045 | ~1/5 = 0.2 |

Discrimination: Standard 0.18 vs Progressive 0.8.

In [ ]:
import sys
sys.path.insert(0, '.')

import torch
import numpy as np
import matplotlib.pyplot as plt
from gfn.reward import HammingReward, ProgressiveHammingReward

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

## Target sequences

In [ ]:
target_sequences = [
    list('UGAGGUAGUAGGUUGUAUAGUU'),  # let-7a
    list('UGAGGUAGUAGGUUGUGUGGUU'),  # let-7b  
    list('UGAGGUAGUACGUUGUAUUGUU'),  # let-7c
]

alphabet = ['A', 'U', 'G', 'C']

for i, seq in enumerate(target_sequences):
    print(f"  {i+1}. {''.join(seq)}")

## Initialize reward functions

In [ ]:
standard_reward = HammingReward(
    target_sequences=target_sequences,
    alphabet=alphabet,
    r_min=0.01,
    device=device
)

progressive_reward = ProgressiveHammingReward(
    target_sequences=target_sequences,
    alphabet=alphabet,
    r_min=0.01,
    device=device,
    prefix_boost=0.5,
    prefix_decay=0.2,
)

## Fake trajectory comparison

Simulate a full 22-step generation: a "good" trajectory that matches the target vs a "bad" one with a random first half.

In [ ]:
target = 'UGAGGUAGUAGGUUGUAUAGUU'  # 22bp target

# Good trajectory: builds the correct sequence step by step
good_trajectory = [target[:i+1] for i in range(len(target))]

# Bad trajectory: first 11bp random, last 11bp happen to match
bad_prefix = 'ACGUACGUACG'  # 11bp random
bad_trajectory = [bad_prefix[:i+1] if i < 11 else bad_prefix + target[11:i+1] for i in range(len(target))]

print(f"Target: {target}")
print(f"\nGood trajectory (correct at every step):")
for i in [0, 4, 10, 15, 21]:
    print(f"  t={i+1:2d}: {good_trajectory[i]}")
    
print(f"\nBad trajectory (random first half, lucky second half):")
for i in [0, 4, 10, 15, 21]:
    print(f"  t={i+1:2d}: {bad_trajectory[i]}")

In [ ]:
steps = list(range(1, 23))

std_good_rewards = [standard_reward(list(good_trajectory[i])) for i in range(22)]
std_bad_rewards = [standard_reward(list(bad_trajectory[i])) for i in range(22)]

prog_good_rewards = [progressive_reward(list(good_trajectory[i]), use_progressive=True) for i in range(22)]
prog_bad_rewards = [progressive_reward(list(bad_trajectory[i]), use_progressive=True) for i in range(22)]

print(f"{'Step':<6} | {'Good Traj':<22} | {'Std Good':>10} | {'Prog Good':>10} | {'Std Bad':>10} | {'Prog Bad':>10}")
print("-"*90)

for i in [0, 2, 5, 10, 15, 21]:
    good_seq = good_trajectory[i][:18] + '...' if len(good_trajectory[i]) > 18 else good_trajectory[i]
    print(f"  {i+1:<4} | {good_seq:<22} | {std_good_rewards[i]:>10.4f} | {prog_good_rewards[i]:>10.4f} | {std_bad_rewards[i]:>10.4f} | {prog_bad_rewards[i]:>10.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax1 = axes[0]
ax1.plot(steps, std_good_rewards, 'g-o', label='Good Trajectory (Correct)', linewidth=2, markersize=4)
ax1.plot(steps, std_bad_rewards, 'r-s', label='Bad Trajectory (First Half Random)', linewidth=2, markersize=4)
ax1.fill_between(steps, std_good_rewards, std_bad_rewards, alpha=0.2, color='blue')
ax1.set_xlabel('Generation Step (t)', fontsize=12)
ax1.set_ylabel('Intermediate Reward', fontsize=12)
ax1.set_title('Standard HammingReward\n(Normalized: matches / target_len)', fontsize=13)
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)
ax1.set_xlim(1, 22)
ax1.set_ylim(0, 1.1)

std_diff_area = np.trapz([g - b for g, b in zip(std_good_rewards, std_bad_rewards)], steps)
ax1.text(12, 0.05, f'Discrimination Area: {std_diff_area:.2f}', fontsize=11, 
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

ax2 = axes[1]
ax2.plot(steps, prog_good_rewards, 'g-o', label='Good Trajectory (Correct)', linewidth=2, markersize=4)
ax2.plot(steps, prog_bad_rewards, 'r-s', label='Bad Trajectory (First Half Random)', linewidth=2, markersize=4)
ax2.fill_between(steps, prog_good_rewards, prog_bad_rewards, alpha=0.2, color='blue')
ax2.set_xlabel('Generation Step (t)', fontsize=12)
ax2.set_ylabel('Intermediate Reward', fontsize=12)
ax2.set_title('ProgressiveHammingReward\n(Normalized: matches / current_len)', fontsize=13)
ax2.legend(loc='lower right')
ax2.grid(True, alpha=0.3)
ax2.set_xlim(1, 22)
ax2.set_ylim(0, 1.1)

prog_diff_area = np.trapz([g - b for g, b in zip(prog_good_rewards, prog_bad_rewards)], steps)

plt.tight_layout()
plt.show()
print(f"\nDiscrimination Area Improvement: {prog_diff_area / std_diff_area:.1f}x")

In [ ]:
# log(reward) matters for FL-DB loss:
# (log F(s_t) + log P_F - log F(s_{t+1}) - log P_B - log R(s_t))^2

log_std_good = [np.log(max(r, 0.001)) for r in std_good_rewards]
log_std_bad = [np.log(max(r, 0.001)) for r in std_bad_rewards]
log_prog_good = [np.log(max(r, 0.001)) for r in prog_good_rewards]
log_prog_bad = [np.log(max(r, 0.001)) for r in prog_bad_rewards]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax1 = axes[0]
ax1.plot(steps, log_std_good, 'g-o', label='Good Trajectory', linewidth=2, markersize=4)
ax1.plot(steps, log_std_bad, 'r-s', label='Bad Trajectory', linewidth=2, markersize=4)
ax1.fill_between(steps, log_std_good, log_std_bad, alpha=0.2, color='blue')
ax1.set_xlabel('Generation Step (t)', fontsize=12)
ax1.set_ylabel('log(Intermediate Reward)', fontsize=12)
ax1.set_title('Standard: log(R) in FL-DB Loss', fontsize=13)
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

ax2 = axes[1]
ax2.plot(steps, log_prog_good, 'g-o', label='Good Trajectory', linewidth=2, markersize=4)
ax2.plot(steps, log_prog_bad, 'r-s', label='Bad Trajectory', linewidth=2, markersize=4)
ax2.fill_between(steps, log_prog_good, log_prog_bad, alpha=0.2, color='blue')
ax2.set_xlabel('Generation Step (t)', fontsize=12)
ax2.set_ylabel('log(Intermediate Reward)', fontsize=12)
ax2.set_title('Progressive: log(R) in FL-DB Loss', fontsize=13)
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics

early_std_diff = np.mean([std_good_rewards[i] - std_bad_rewards[i] for i in range(5)])
early_prog_diff = np.mean([prog_good_rewards[i] - prog_bad_rewards[i] for i in range(5)])
print(f"Mean reward gap (first 5 steps):")
print(f"  Standard:    {early_std_diff:.4f}")
print(f"  Progressive: {early_prog_diff:.4f}")
print(f"  Improvement: {early_prog_diff / early_std_diff:.1f}x")

total_log_std_diff = sum(log_std_good) - sum(log_std_bad)
total_log_prog_diff = sum(log_prog_good) - sum(log_prog_bad)
print(f"\nCumulative log(R) gap over full trajectory:")
print(f"  Standard:    {total_log_std_diff:.2f}")
print(f"  Progressive: {total_log_prog_diff:.2f}")
print(f"  Improvement: {total_log_prog_diff / total_log_std_diff:.1f}x")

## Prefix boost effect

In [ ]:
prog_no_boost = ProgressiveHammingReward(
    target_sequences=target_sequences,
    alphabet=alphabet,
    r_min=0.01,
    device=device,
    prefix_boost=0.0,
)

prog_with_boost = ProgressiveHammingReward(
    target_sequences=target_sequences,
    alphabet=alphabet,
    r_min=0.01,
    device=device,
    prefix_boost=1.0,
    prefix_decay=0.15,
)

In [ ]:
# Prefix match vs suffix match with same number of matching positions
# Target: UGAGGUAGUAGGUUGUAUAGUU (22bp)

prefix_match = list('UGAGGUAGUAG' + 'AAAAAAAAAAA')  # first 11 match
suffix_match = list('AAAAAAAAAAA' + 'GUUGUAUAGUU')  # last 11 match

print(f"Target:       UGAGGUAGUAGGUUGUAUAGUU")
print(f"Prefix match: {''.join(prefix_match)} (first 11 match)")
print(f"Suffix match: {''.join(suffix_match)} (last 11 match)")

r_prefix_no_boost = prog_no_boost(prefix_match, use_progressive=True)
r_suffix_no_boost = prog_no_boost(suffix_match, use_progressive=True)

r_prefix_with_boost = prog_with_boost(prefix_match, use_progressive=True)
r_suffix_with_boost = prog_with_boost(suffix_match, use_progressive=True)

print(f"\nWithout prefix_boost (=0):")
print(f"  Prefix match: {r_prefix_no_boost:.4f}")
print(f"  Suffix match: {r_suffix_no_boost:.4f}")
print(f"  Diff: {r_prefix_no_boost - r_suffix_no_boost:+.4f}")
print(f"\nWith prefix_boost (=1.0):")
print(f"  Prefix match: {r_prefix_with_boost:.4f}")
print(f"  Suffix match: {r_suffix_with_boost:.4f}")
print(f"  Diff: {r_prefix_with_boost - r_suffix_with_boost:+.4f}")

## Summary

| Feature | Standard HammingReward | ProgressiveHammingReward |
|---------|----------------------|--------------------------|
| Intermediate normalization | `matches / target_len` | `matches / current_len` |
| Early-step discrimination | Low (~0.05) | High (~1.0) |
| Prefix boost | No | Optional (extra weight on early positions) |

For FL-DB training with `insert_only` mode, use `ProgressiveHammingReward` with `prefix_boost` in the 0.3-1.0 range.